# TabPFN → drzewo decyzyjne (rozwiązanie 2)

Colab: **Runtime → Change runtime type → T4 GPU**, potem Run all.

Pipeline destylacji:

1. **TabPFN** trenuje się wyłącznie na dozwolonym `val.csv` (holdout i test nie wchodzą).
2. Nauczyciel **dopisuje etykiety** do nieoznaczonego `train.csv` (pseudo-labelki).
3. **Student (drzewo)** uczy się ze znacznie większego zbioru: `val` ∪ `train`.

W notebooku są **dwa** drzewa na tym samym nauczycielu, żeby porównać z wariantem „tylko val”. Aplikacja (`app_tabpfn.py`) dostaje drzewo z powiększonego zbioru — CPU, ścieżka if/then.

**Weryfikacja:** uczciwy Raw_Score na `final_valid.csv` (silniki, których model nie widział). `test.csv` zostaje submitem bez etykiet.

In [1]:
import sys
from pathlib import Path


IN_COLAB = "google.colab" in sys.modules
IN_COLAB = False
if IN_COLAB:
    %pip install -q tabpfn scikit-learn pandas numpy torch joblib plotly
    from google.colab import files
    print("Wgraj val.csv, final_valid.csv, train.csv, test.csv oraz tabpfn_diagnose.py")
    files.upload()

import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

cuda: False CPU


In [2]:
from tabpfn_diagnose import TabPFNTreeDiagnoser, hackathon_score, pick_device, print_eval, read_labeled_csv
import pandas as pd

val = read_labeled_csv("val.csv")
holdout = read_labeled_csv("final_valid.csv")
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
overlap = set(val["engine_id"]) & set(holdout["engine_id"])
assert not overlap, f"wyciek silników val ∩ final_valid: {sorted(overlap)}"
print(
    "val", val.shape, "final_valid", holdout.shape,
    "train", train.shape, "test", test.shape,
    "device=", pick_device(),
)
print("silniki holdout:", sorted(holdout["engine_id"].unique()))

val (400, 26) final_valid (76, 26) train (2400, 24) test (600, 24) device= cpu
silniki holdout: ['val_0033', 'val_0034', 'val_0035', 'val_0036', 'val_0037', 'val_0038', 'val_0039']


## Nauczyciel: TabPFN na `val.csv`

TabPFN widzi wyłącznie dozwolony valid. Najpierw destylujemy drzewo **tylko z val** (dotychczasowy student) — holdout zapisujemy do porównania. Na T4 możesz zostawić `do_cv=True` (GroupKFold po silniku). Na CPU: `do_cv=False`.

In [3]:
device = pick_device()
n_estimators = 8 if device == "cuda" else 4
do_cv = device == "cuda"  # T4: extra GroupKFold na val; CPU: pomiń
CONF_MIN = 0.70  # pewność pseudo-etykiet; 0.0 = weź cały train.csv

model = TabPFNTreeDiagnoser().fit(
    val,
    train=None,  # student A: tylko val
    device=device,
    n_estimators=n_estimators,
    do_cv=do_cv,
)
sub_ho_tree_val_only = model.predict(holdout)
print("student A (tylko val)  n_student=", model.meta.get("n_student"))
print(model.meta)

TabPFN teacher  device=cpu  n_estimators=4
TabPFN in-sample on val.csv (sanity, not the holdout score):
              precision    recall  f1-score   support

          ok      1.000     1.000     1.000       345
 zakoksowany      1.000     1.000     1.000        17
      lejacy      1.000     1.000     1.000        11
       pompa      1.000     1.000     1.000         7
      iglica      1.000     1.000     1.000        10
     unknown      1.000     1.000     1.000        10

    accuracy                          1.000       400
   macro avg      1.000     1.000     1.000       400
weighted avg      1.000     1.000     1.000       400

Student tree distilled on 400 rows (val=400, pseudo=0)

Distilled tree on val.csv (resubstitution, not holdout):
              precision    recall  f1-score   support

          ok      1.000     0.997     0.999       345
 zakoksowany      1.000     1.000     1.000        17
      lejacy      1.000     1.000     1.000        11
       pompa      0.833

## Student z powiększonym zbiorem: `val` ∪ `train`

TabPFN (już wytrenowany) etykietuje `train.csv`. Drzewo destylujemy ze **znacznie większego** zbioru: prawdziwe labelki z val + pseudo-labelki z train. Nauczyciel się nie trenuje drugi raz.

Porównanie na `final_valid.csv`: teacher / drzewo←val / drzewo←val+train. Do aplikacji zapisujemy wariant z train.

In [4]:
y_ho = holdout["label"].to_numpy()
s_ho = holdout["severity"].to_numpy()
sub_ho_tabpfn = model.predict_teacher(holdout, model.teacher_)
n_fit_val = int(model.meta["n_student"])

model.distill(train, conf_min=CONF_MIN)  # student B: val + pseudo-train
sub_ho_tree = model.predict(holdout)
n_fit_big = int(model.meta["n_student"])
model.save()

rows = []
for name, sub, n_fit in [
    ("TabPFN teacher", sub_ho_tabpfn, len(val)),
    ("drzewo ← tylko val", sub_ho_tree_val_only, n_fit_val),
    ("drzewo ← val + pseudo-train", sub_ho_tree, n_fit_big),
]:
    raw, macro, sev = hackathon_score(
        y_ho, sub["label"].to_numpy(), s_ho, sub["severity"].to_numpy()
    )
    agree = float((sub["label"].to_numpy() == sub_ho_tabpfn["label"].to_numpy()).mean())
    rows.append(
        {
            "model": name,
            "n_fit": n_fit,
            "Raw_Score": raw,
            "macro-F1": macro,
            "severity_acc": sev,
            "zgoda vs TabPFN": agree,
        }
    )
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print(
    f"\npseudo-labelki: {model.meta.get('n_pseudo')}/{len(train)} "
    f"(próg P ≥ {CONF_MIN:.2f})  →  student {n_fit_val} → {n_fit_big} wierszy"
)
print("rozkład pseudo-etykiet:", model.meta.get("train_pseudo", {}).get("label_counts"))

Pseudo-labels from train.csv: 2359/2400 with max P ≥ 0.70
ok             2035
zakoksowany      73
iglica           65
unknown          64
lejacy           62
pompa            60
Student tree distilled on 2759 rows (val=400, pseudo=2359)

Distilled tree on val.csv (resubstitution, not holdout):
              precision    recall  f1-score   support

          ok      1.000     1.000     1.000       345
 zakoksowany      1.000     1.000     1.000        17
      lejacy      0.917     1.000     0.957        11
       pompa      1.000     0.857     0.923         7
      iglica      1.000     1.000     1.000        10
     unknown      1.000     1.000     1.000        10

    accuracy                          0.998       400
   macro avg      0.986     0.976     0.980       400
weighted avg      0.998     0.998     0.997       400

tree vs val labels  macro-F1=0.9799  severity_acc=1.0000  Raw_Score=0.9849  fidelity vs TabPFN=0.998
Wrote /home/janek/Desktop/hackathon-engin/artifacts/diagnoser

## Holdout: `final_valid.csv` — raporty klas

Silniki spoza `val.csv`. Poniżej pełny classification_report dla nauczyciela i **wybranego** studenta (val + pseudo-train). Tabelka porównawcza jest w komórce wyżej.

In [5]:
print_eval("final_valid — TabPFN teacher", y_ho, sub_ho_tabpfn["label"].to_numpy(), s_ho, sub_ho_tabpfn["severity"].to_numpy())
print_eval("final_valid — drzewo ← val + pseudo-train", y_ho, sub_ho_tree["label"].to_numpy(), s_ho, sub_ho_tree["severity"].to_numpy())
print_eval("final_valid — drzewo ← tylko val", y_ho, sub_ho_tree_val_only["label"].to_numpy(), s_ho, sub_ho_tree_val_only["severity"].to_numpy())
print(
    f"zgoda drzewo(val+train) vs TabPFN: {(sub_ho_tree['label'] == sub_ho_tabpfn['label']).mean():.3f}  |  "
    f"zgoda drzewo(tylko val) vs TabPFN: {(sub_ho_tree_val_only['label'] == sub_ho_tabpfn['label']).mean():.3f}"
)


=== final_valid — TabPFN teacher ===
              precision    recall  f1-score   support

          ok      1.000     1.000     1.000        62
 zakoksowany      1.000     1.000     1.000         1
      lejacy      1.000     1.000     1.000         3
       pompa      1.000     1.000     1.000         2
      iglica      1.000     1.000     1.000         6
     unknown      1.000     1.000     1.000         2

    accuracy                          1.000        76
   macro avg      1.000     1.000     1.000        76
weighted avg      1.000     1.000     1.000        76

macro-F1=1.0000  severity_acc=0.9167  Raw_Score=0.9792

=== final_valid — drzewo ← val + pseudo-train ===
              precision    recall  f1-score   support

          ok      1.000     1.000     1.000        62
 zakoksowany      0.500     1.000     0.667         1
      lejacy      1.000     1.000     1.000         3
       pompa      0.500     0.500     0.500         2
      iglica      1.000     0.833     0.90

## Drzewo, które zobaczy mechanik

In [6]:
print(model.rules_text())

|--- residual 9 kHz vs. baseline silnika <= -8.43
|   |--- residual 13 kHz vs. baseline silnika <= 4.53
|   |   |--- podobieństwo do wzorca: zakoksowany <= -0.30
|   |   |   |--- residual 15 kHz vs. baseline silnika <= -15.37
|   |   |   |   |--- class: lejacy
|   |   |   |--- residual 15 kHz vs. baseline silnika >  -15.37
|   |   |   |   |--- class: lejacy
|   |   |--- podobieństwo do wzorca: zakoksowany >  -0.30
|   |   |   |--- podobieństwo do wzorca: iglica <= 0.93
|   |   |   |   |--- podobieństwo do wzorca: pompa <= 0.93
|   |   |   |   |   |--- odchyłka L1 od profilu silnika <= 176.74
|   |   |   |   |   |   |--- class: ok
|   |   |   |   |   |--- odchyłka L1 od profilu silnika >  176.74
|   |   |   |   |   |   |--- class: unknown
|   |   |   |   |--- podobieństwo do wzorca: pompa >  0.93
|   |   |   |   |   |--- class: pompa
|   |   |   |--- podobieństwo do wzorca: iglica >  0.93
|   |   |   |   |--- odbicie 9 → 12 kHz <= 2.02
|   |   |   |   |   |--- class: pompa
|   |   |   |

## Submit `test.csv` + zgodność nauczyciel / student

Ten sam nauczyciel (fit na `val.csv`) i student z `val` ∪ `train`. `test.csv` nie ma etykiet.

In [7]:
sub_tree = model.predict(test)
sub_tabpfn = model.predict_teacher(test, model.teacher_)
sub_tree.to_csv("predictions_tree.csv", index=False)
sub_tabpfn.to_csv("predictions_tabpfn.csv", index=False)
agree = (sub_tree["label"] == sub_tabpfn["label"]).mean()
print(f"zgoda drzewo vs TabPFN na teście (bez etykiet): {agree:.3f}")
print("TabPFN\n", sub_tabpfn["label"].value_counts())
print("drzewo\n", sub_tree["label"].value_counts())

if IN_COLAB:
    files.download("predictions_tabpfn.csv")
    files.download("predictions_tree.csv")
    files.download("artifacts/diagnoser_tree.joblib")

zgoda drzewo vs TabPFN na teście (bez etykiet): 0.990
TabPFN
 label
ok             518
unknown         21
zakoksowany     16
pompa           16
iglica          16
lejacy          13
Name: count, dtype: int64
drzewo
 label
ok             518
unknown         24
zakoksowany     17
pompa           15
lejacy          13
iglica          13
Name: count, dtype: int64


Lokalnie po pobraniu `diagnoser_tree.joblib` do `artifacts/`:

```bash
streamlit run app_tabpfn.py
```